## 1. Environment and Imports

This section imports the libraries used for document loading, legal-text processing,
chunking, embeddings, vector storage, and environment configuration.

The project uses Chroma as the vector database and Hugging Face MiniLM embeddings
for semantic retrieval.

In [1]:
import os
import glob
import re
from pathlib import Path
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 2. Configuration

This cell defines the language model and the local Chroma vector-store directory.

Environment variables are loaded from the `.env` file.

In [2]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

load_dotenv(override = True)

True

## 3. Loading the EU AI Act Corpus

The EU AI Act knowledge base is stored as Markdown files grouped into Articles,
Recitals, and Annexes.

Each document is loaded and assigned a `doc_type` metadata field indicating its
legal document category.

In [3]:
folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob = "**/*.md",
        loader_cls = TextLoader,
        loader_kwargs = {"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 306 documents


## 4. Legal Metadata Extraction and Chunking

Before splitting the documents into chunks, legal identifiers are extracted from
the filenames.

For Articles, the Article number and title are added to the document metadata.
Recitals and Annexes receive their corresponding legal identifiers.

The documents are then divided into overlapping chunks. Each chunk is prefixed
with its legal identifier so that the embedding model retains information about
the provision from which the text originates.

In [4]:
# Add legal identifiers and titles BEFORE splitting
for doc in documents:
    source = doc.metadata.get("source", "")
    filename = os.path.basename(source)

    identifier = ""
    title = ""

    article_match = re.match(r"article_(\d+)", filename)
    recital_match = re.match(r"recital_(\d+)", filename)
    annex_match = re.match(r"annex_([IVXLCDM]+)", filename)

    if article_match:
        number = int(article_match.group(1))
        identifier = f"Article {number}"

        # Extract the title appearing after "Article N"
        title_match = re.search(
            rf"## Official text\s*\n+\s*Article\s+{number}\s*\n+\s*([^\n]+)",
            doc.page_content
        )

        if title_match:
            title = title_match.group(1).strip()

    elif recital_match:
        identifier = f"Recital {int(recital_match.group(1))}"

    elif annex_match:
        identifier = f"Annex {annex_match.group(1)}"

    doc.metadata["identifier"] = identifier
    doc.metadata["title"] = title


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)


# Add legal identity to EVERY chunk
for chunk in chunks:
    identifier = chunk.metadata.get("identifier", "")
    title = chunk.metadata.get("title", "")

    if title:
        header = f"{identifier} — {title}"
    else:
        header = identifier

    chunk.page_content = (
        f"{header}\n\n"
        f"{chunk.page_content}"
    )


print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 1060 chunks
First chunk:

page_content='Annex I

# Annex I

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

ANNEX I	 
List of Union harmonisation legislation
Section A.' metadata={'source': 'knowledge-base\\annexes\\annex_I.md', 'doc_type': 'annexes', 'identifier': 'Annex I', 'title': ''}


## 5. Embeddings and Vector Store

Each legal-text chunk is converted into a semantic embedding using
`all-MiniLM-L6-v2`.

The embeddings are stored in a persistent Chroma vector database. If an existing
vector store is present, it is recreated so that the database reflects the
current chunking and metadata configuration.

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

if os.path.exists(db_name):
    Chroma(
        persist_directory = db_name,
        embedding_function = embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

Vectorstore created with 1060 documents


## 6. Vector Store Inspection

This diagnostic cell checks the number and dimensionality of the embeddings
stored in Chroma.

The MiniLM embedding model produces 384-dimensional vectors.

In [6]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(
    limit = 1,
    include = ["embeddings"]
)["embeddings"][0]

dimensions = len(sample_embedding)

print(
    f"There are {count:,} vectors with "
    f"{dimensions:,} dimensions in the vector store"
)

There are 1,060 vectors with 384 dimensions in the vector store


## 7. Retrieval and Generation Dependencies

This section imports the components used for language-model generation,
message construction, and the interactive Gradio interface.

The retrieved EU AI Act context will be supplied to an OpenAI language model
to generate grounded answers.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

## 8. Retriever and Language Model

The Chroma vector store is configured as a retriever that returns the 10 most
relevant chunks for each query.

GPT-4.1-nano is used as the generation model with a temperature of 0 to promote
consistent and deterministic answers.

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

llm = ChatOpenAI(
    temperature = 0,
    model_name = MODEL
)

## 9. Legal RAG Prompt

The production system prompt is defined in `implementation/answer.py`.

It constrains the language model to answer from the retrieved EU AI Act context,
identify the main relevant Article, Recital, or Annex, distinguish general rules
from exceptions and conditions, and avoid presenting generated responses as
legal advice.

Keeping the prompt in the shared implementation ensures that the notebook and
Gradio application use the same legal-answering behavior.

## 10. Direct Provision Retrieval

The shared implementation supports direct lookup of specific legal provisions,
including Articles, Recitals, and Annexes.

Queries such as `Article 6?`, `Recital 47?`, or `Annex I?` bypass semantic
retrieval and return the corresponding source document directly.

This functionality is implemented in `implementation/answer.py` so that the
notebook and Gradio application use the same provision-lookup logic.

## 11. Question-Answering Pipeline

The main question-answering function combines direct Article lookup with the
standard RAG workflow.

For ordinary questions, the query is sent to the Chroma retriever, the retrieved
legal chunks are assembled as context, and the language model generates an answer
under the constraints of the legal RAG prompt.

The response also includes a list of unique legal provisions retrieved as
references.

In [9]:
import importlib
import implementation.answer

importlib.reload(implementation.answer)

from implementation.answer import answer_question

## 12. Example Query

The following example tests the complete RAG pipeline with a question about
the application date of the EU AI Act.

The expected primary legal source is Article 113, which specifies the general
application date as well as provisions with earlier or later application dates.

In [10]:
answer, docs = answer_question(
    "When does the EU AI Act apply?",
    []
)

print(answer)

According to Article 113, the EU AI Act shall apply from 2 August 2026. However, there are specific provisions that apply earlier: Chapters I and II from 2 February 2025; Chapter III Section 4, Chapter V, Chapter VII, Chapter XII, and Article 78 from 2 August 2025 (with the exception of Article 101); and Article 6(1) and the corresponding obligations from 2 August 2027.

### References
- Article 113 — Entry into force and application
- Article 110 — Amendment to Directive (EU) 2020/1828
- Article 105 — Amendment to Directive 2014/90/EU
- Article 108 — Amendments to Regulation (EU) 2018/1139
- Article 109 — Amendment to Regulation (EU) 2019/2144
- Article 52 — Procedure
- Article 64 — AI Office
- Article 106 — Amendment to Directive (EU) 2016/797
- Article 104 — Amendment to Regulation (EU) No 168/2013
- Article 107 — Amendment to Regulation (EU) 2018/858


## 13. Retrieval Diagnostic: Application Date

This diagnostic examines the top 10 chunks retrieved for a question about
when the EU AI Act applies.

Article 113 should rank highly because it contains the Regulation's entry-into-force
and application provisions. Inspecting the remaining results also helps identify
irrelevant chunks introduced by dense semantic retrieval.

In [11]:
docs = retriever.invoke(
    "When does the EU AI Act apply?"
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])


--- Result 1 ---
{'title': 'Entry into force and application', 'doc_type': 'articles', 'identifier': 'Article 113', 'source': 'knowledge-base\\articles\\article_113.md'}
Article 113 — Entry into force and application

# Article 113

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

Article 113 
Entry into force and application
This Regulation shall enter into force on the twentieth day following that of 
its publication in the Official Journal of the European Union.
It shall apply from 2 August 2026. However:
(a)	 Chapters I and II shall apply from 2 February 2025;
(b)	 Chapter III Section 4, Chapter V, Chapter VII and Chapter XII and 
Article 78 shall apply from 2 August 2025, with the exception of 
Article 101;
(c)	 Article 6(1) and the corresponding obligations in this Regulation shall 
apply from 2 August 2027.

This Regulation shall be binding in its entirety and directly applicable in all 
Member States. Done at Br

--- Result 2 ---
{'identifi

## 14. Retrieval Diagnostic: Prohibited AI Practices

This example tests retrieval for a broader legal question concerning prohibited
AI practices.

Article 5 is correctly retrieved as the top result. However, several less relevant
provisions also appear among the top results, illustrating a limitation of the
baseline dense-retrieval approach.

In [12]:
docs = vectorstore.similarity_search(
    "Which AI practices are prohibited under the EU AI Act?",
    k = 10
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:700])


--- Result 1 ---
{'identifier': 'Article 5', 'title': 'Prohibited AI practices', 'doc_type': 'articles', 'source': 'knowledge-base\\articles\\article_005.md'}
Article 5 — Prohibited AI practices

# Article 5

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

--- Result 2 ---
{'source': 'knowledge-base\\articles\\article_108.md', 'title': 'Amendments to Regulation (EU) 2018/1139', 'identifier': 'Article 108', 'doc_type': 'articles'}
Article 108 — Amendments to Regulation (EU) 2018/1139

Article 108 
Amendments to Regulation (EU) 2018/1139
Regulation (EU) 2018/1139 is amended as follows:
(1)	
in Article 17, the following paragraph is added:
‘3. Without prejudice to paragraph 2, when adopting implementing acts 
pursuant to paragraph 1 concerning Artificial Intelligence systems which 
are safety components within the meaning of Regulation (EU) 2024/1689 
of the European Parliament and of the Council (*), the requirements set 
out in Chapter III, Section

## 15. Interactive RAG Demo

The RAG pipeline can be tested interactively through a simple Gradio chat
interface.

Questions submitted through the interface are passed to the same
`answer_question` function used in the preceding examples.

In [13]:
def notebook_chat(message, history):
    answer, _ = answer_question(message, history)
    return answer

gr.ChatInterface(
    notebook_chat,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 16. Evaluation Setup

The following section evaluates the EU AI Act RAG using the project's dedicated
evaluation modules and test dataset.

The evaluation framework assesses both retrieval performance and generated-answer
quality using EU AI Act-specific test cases.

### 16.1 Evaluation Functions

The evaluation module provides separate functions for assessing retrieval
performance and generated-answer quality.

Reloading the module ensures that the notebook uses the latest version of the
evaluation implementation.

In [14]:
import evaluation.eval

importlib.reload(evaluation.eval)

from evaluation.eval import evaluate_retrieval, evaluate_answer

### 16.2 Embedding Compatibility Check

The evaluation pipeline uses the same MiniLM embedding model as the RAG system.

This check confirms that the embedding model produces 384-dimensional vectors,
consistent with the vectors stored in Chroma.

In [15]:
from implementation.answer import embeddings

print(len(embeddings.embed_query("test")))


384


### 16.3 Evaluation Dataset

The evaluation dataset contains test questions designed specifically for the
EU AI Act RAG system.

Each test includes a question, category, reference answer, and keywords used
to evaluate retrieval and answer quality.

In [16]:
from evaluation import test
tests = test.load_tests()
len(tests)

20

### 16.4 Evaluation Case Inspection

The following cell inspects one evaluation case to verify its question,
category, reference answer, and expected legal keywords.

In [17]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

What is the purpose of the EU AI Act?
direct_fact
According to Article 1, the purpose of the EU AI Act is to improve the functioning of the internal market and promote the uptake of human-centric and trustworthy artificial intelligence while ensuring a high level of protection of health, safety and fundamental rights and supporting innovation.
['Article 1', 'internal market', 'human-centric', 'trustworthy', 'fundamental rights', 'innovation']


### 16.5 Test Category Distribution

The evaluation cases cover several types of legal questions, including
obligations, multi-part provisions, classification, scope, definitions,
transparency, and temporal application.

The category distribution provides an overview of the legal reasoning tasks
represented in the test set.

In [18]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'obligation': 9,
         'multi_part': 4,
         'classification': 2,
         'direct_fact': 1,
         'scope': 1,
         'definition': 1,
         'transparency': 1,
         'temporal': 1})

### 16.6 Retrieval Evaluation

Retrieval quality is evaluated using Mean Reciprocal Rank (MRR), normalized
Discounted Cumulative Gain (nDCG), and keyword coverage.

These metrics measure whether legally relevant information is retrieved and
how highly relevant chunks are ranked.

In [19]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.08333333333333333, ndcg=0.1781035935540111, keywords_found=3, total_keywords=6, keyword_coverage=50.0)

### 16.7 Answer Evaluation

Answer quality is evaluated by comparing the generated response with the
reference answer.

The evaluation considers accuracy, completeness, relevance, and legal-source
accuracy. Legal-source accuracy specifically assesses whether the generated
answer identifies the appropriate EU AI Act provision.

In [20]:
eval, answer, chunks = evaluate_answer(example)

In [21]:
eval

AnswerEval(feedback='The answer correctly states the purpose of the EU AI Act, accurately citing Recital 1 and covering the main objectives such as promoting trustworthy AI and protecting fundamental rights. It also expands on supporting innovation and facilitating cross-border AI development, which is relevant and valuable. However, it omits mention of improving the internal market, which is a key aspect highlighted in the reference answer. The response is comprehensive in its coverage but could be slightly more aligned with the reference focus on the internal market.', accuracy=4.0, completeness=4.0, relevance=5.0, legal_source_accuracy=5.0)

### 16.8 Evaluation Feedback

The evaluator also provides qualitative feedback explaining the assigned
scores. This helps identify specific weaknesses in generated answers that
may not be apparent from aggregate numerical metrics alone.

In [22]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)
print(eval.legal_source_accuracy)

The answer correctly states the purpose of the EU AI Act, accurately citing Recital 1 and covering the main objectives such as promoting trustworthy AI and protecting fundamental rights. It also expands on supporting innovation and facilitating cross-border AI development, which is relevant and valuable. However, it omits mention of improving the internal market, which is a key aspect highlighted in the reference answer. The response is comprehensive in its coverage but could be slightly more aligned with the reference focus on the internal market.
4.0
4.0
5.0
5.0
